In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧱 W14-D3 · 实验 1：business-ontology.yaml 机器体检——语义资产半年后会变成什么

> 开发期 · Week 14「语义资产工程化」Day 3（2026-09-02 周三）
> 昨日 D2 裁定了分层 SoT（ontology 是词汇层权威、Semantic Model 是对账层），今天兑现预告：给这份唯一权威做一次全量机器体检，并把手工映射表程序化。
> 配套实验：`Day3-OntologyYaml解析与校验.ipynb`（6 个 cell 全部可执行，产物含机器可读体检报告 `w14d3-ontology-health-report.yaml`）

---


## 0. 今日开发目标


In [ ]:
① PyYAML 解析 business-ontology.yaml，schema 形状校验 + 完整性剖面（按模块）
② 术语质量审计：度分布 / 跨模块歧义 / 别名撞术语
③ capability 锚点审计：L 占位 vs 命名命中，按模块拆开（D2 只有全局数）
④ 模块×Context 映射程序化（替代 D2 手工 v0.1）+ 孤儿 Context + 报告落盘
⑤ 接入 PT-W4-D6 产出：六条装配规则 W1-W6 对 ontology 判分（D1 遗留的"PT-W4 接入"在此兑现）
⑥ 用熵增模拟回答 Today's Question


---


## 1. Today's Question：语义资产如果机器不可校验，半年后会变成什么？

**一句话答案：它不会"写乱"，它会"过期"——而且过得很安静。烂的方式不是格式崩坏（今天实测 schema 形状 102/102 规整、零意外键），而是三种熵在无人观测的情况下单调增长：完整性熵（role 7/102、场景 15/102 停更）、歧义熵（37 个跨模块术语无人消歧）、锚点熵（spec 增长下 12% 反向覆盖持续稀释，模拟半年 → 9%）。无人治理 + 机器不可校验 = 熵增不可见；熵增不可见 = 半年后它就是另一份 Word 数据字典。**

### 1.1 体检五层结果（全部今日实测，ipynb 可复现）

| 体检层 | 结果 | 判定 |
|---|---|---|
| 治理 | 顶层键只有 `modules:`，无 frontmatter（版本/状态/维护人/变更流程全缺） | ❌ 零治理 |
| 结构形状 | 12 模块 / 102 子功能，键集完全符合 `{capabilities, role, scenarios, terms}`，**意外键 0** | ✅ 纪律好 |
| 完整性 | role 7/102（6.9%）、场景 15/102——**全部集中在资源管理 7 个子功能**，其余 11 模块 0 条 | ⚠️ 尾部烂尾 |
| 术语质量 | 883 条目 → **792 唯一术语**（91 条复用登记）；65 个术语多处出现（55×2 / 6×3 / 3×4 / **「项目」×16**）；**37 个跨模块歧义**；模块别名撞他模块术语 6 处 | ⚠️ 无消歧机制 |
| capability 锚点 | 102 标签 = 23 L 占位 + 79 命名；命中 spec 33（42%）；反向 33/273（12%）；跨模块复用标签 12 个（platform-foundation 挂 **7 模块 33 子功能**） | ⚠️ 贴纸非锚点 |

**D2 的数字今天全部机器复核通过**（12/102/883/15/23/42%/12%），另有两处 D2 没有的新发现：

1. **"883 术语"是条目数，唯一术语只有 792**——91 条复用登记本来是设计允许（术语跨场景），但没有消歧声明，「项目」「商户」「出账」这类跨模块词喂给 LLM 就是歧义输入。最极端的「项目」被登记在 16 个子功能、跨 5 个模块。
2. **贴纸质量按模块极不均匀**：财务管理 12/12 全锚定（100%），而移动端 1/15（7%）、数据决策 2/14（14%）、预算管理 1/8（12%）、运营管理 2/8（25%）。**越是近期扩张的模块，贴纸越失真**——这正是"锚点熵"正在发生的直接证据。

### 1.2 PT-W4 接入：六条装配规则判分 0/6

把 PT-W4-D6 发明的六条装配规则（符号解析级校验）逐条对 ontology 求值：

| 规则 | 要求 | ontology 现状 | 判定 |
|---|---|---|---|
| W1 谓词可解析 | Rule 谓词名词 ∈ Entity 定义 | 无 Entity/Rule 构件 | N/A-fail |
| W2 挂载点存在 | Guard 挂在已声明迁移上 | 无 Lifecycle 构件 | N/A-fail |
| W3 类型封闭 | effects ∈ 冻结注册表 5 类 | 无 Event 构件 | N/A-fail |
| W4 依赖可解析 | capability_dependencies ∈ Capability Map | 无依赖/状态构件 | N/A-fail |
| W5 边界包含 | Agent 认知边界 ⊆ 17 Context | 无 Agent Card 构件 | N/A-fail |
| W6 变体锚定 | Variant 规则声明 base | 无 Rule Card 构件 | N/A-fail |

0/6 不是差评，是**定位确认**：词汇层本来就不该有这六种构件——但 Semantic Model 必须有，因为消费方（数字员工 / LnkChatBI 的 NL→SQL 校准）需要的是 W1-W6 能跑起来的语义层。**这就是"词汇认识对象、语义才认识事实"的机器判定版。**

### 1.3 ERP 人话（26 年对照）

传统 ERP 里这份文件叫**数据字典**（Data Dictionary）。每家公司的数据字典都死于同一种方式：上线时是准的，三个月后加了字段没人改，半年后新人发现"字典和库对不上"，一年后大家默契地不再打开它——**它变成了一份需要被"考古"的文档，而考古成本高到不如直接读库**。数据字典从来不是被推翻的，是被稀释死的。

今天模拟出的两条曲线就是稀释过程的量化：lnkcre 的 specs 每月都在涨，ontology 的引用冻结在 33——**被对账的一侧在动、对账的依据不动，覆盖率 12%→9% 的稀释不需要任何人犯错**。唯一的差别是：26 年前我们没有 CI，现在有——给语义资产做 schema 校验 + W1-W6 判分 + 度量报告，成本约等于今天这 150 行 Python。

---


## 2. 完成事实

### 2.1 产物清单（全部落盘第14周/）

| 产物 | 说明 |
|---|---|
| `Day3-OntologyYaml解析与校验.ipynb` | 5 组实验，verify 通过（6 code cells OK） |
| `w14d3-ontology-health-report.yaml` | **机器可读体检报告**（含源文件 sha256 基线，D6 定稿包直接原料） |
| `w14d3_模块Context映射.png` | 12×17 映射热力图 + 每模块 spec 锚定数并排 |
| `w14d3_熵增模拟.png` | 无校验线（稀释）vs CI 线（抬升）双曲线 + 歧义熵外推 |

### 2.2 模块×Context 程序化映射 v0.1（替代 D2 手工版）

扇出合计 14：12 模块 → 13/17 Context。**孤儿 Context 4 个：02 Party Core、12 Engineering、15 Customer/Member、16 Parking**——与 D2 反向缺口清单吻合，其中 **02 Party Core 恰是 Gap Analysis 里最重的整改对象（三业态 Party 未建、`normalizeStatus('pending')→'active'` 无待核验态）**：ontology 的盲区和代码的债务重合，不是巧合，是同一份访谈覆盖缺口的两个投影。热力图右侧并排的 spec 锚定数还给出第二个信号：**映射到 Context 多的模块（资源/财务）锚定质量反而最高，无 Context 挂靠的平台类模块（系统管理/资产管理/移动端）靠 platform-foundation 万能贴纸过活**。

### 2.3 Semantic Model 宪章候选条款（从今日证据直接导出）

1. **每个被消费的源必须有 frontmatter**（版本/状态/维护人/变更流程）——今日证据：ontology 治理层 0 分，对账基线无法 pin；
2. **体检必须可复现**：报告携带源文件 hash（今日已做 sha256 前 16 位），半年后重跑同一体检，diff 即熵增量；
3. **术语跨模块复用必须带消歧声明**（语境限定或 scoped alias）——今日证据：37 个跨模块术语 + 「项目」×16。

---


## 3. 遗留 / 风险

- **场景层溯源（明源 6 / 华侨城 12 / 锦和 8 / 悦商 2）**：四家需求方的分布说明资源管理模块的访谈是"四家全收敛"，其余 11 模块访谈纪要是否已丢失（还是从未结构化）未知——D6 定稿时在缺口列表登记。
- **+10 spec/月是假设不是测量**：熵增模拟的绝对值不可引用，方向结论（无校验→稀释）可引用；D5 做 Context 覆盖率时可用 canonical_tables 336 表增长做第二证据源交叉验证。
- 昨日登记的「D1 cron 缺产」补齐路径：今日 §1.2 完成 PT-W4 六构件接入，D1 的"lnkcre 现状对齐"剩余项（R-wave 跟读机制）挪到 D5 开发节奏环节一并处理。


## 4. 明日连接（D4 · Lifecycle/Rule/Policy 层盘点）

对账从"词汇层 vs 语义层"推进到"**声明的规则 vs 代码的 if-else**"：effect-registry.yaml 冻结的 5 类 Lifecycle Effect vs mi 代码事实（lease 状态机、condition-approval、amendment matrix）逐条对账。Today's Question：**语义层声明的规则和代码里的 if-else，谁是 source of truth？**——今天 0/6 的判分说明"声明侧"目前几乎空白，明天开始量"实现侧"到底有多少没被声明。

---

### 附：今日证据清单

| 证据 | 来源 |
|---|---|
| 治理零分 / 形状零违规 / role 7/102 / 场景 15/102（全在资源管理） | ipynb 实验一（PyYAML 解析实测） |
| 883→792 唯一 / 度分布 {1:727, 2:55, 3:6, 4:3, 16:1} / 跨模块 37 / 别名撞术语 6 | ipynb 实验二 |
| 23 L 占位 / 33 命中 = 42% / 反向 33/273 = 12% / 复用 12 个 / platform-foundation×33 / 按模块命中率（财务 100% vs 移动端 7%） | ipynb 实验三 + `/root/lnkcre/openspec/specs/` 目录实测 |
| 扇出 14 / 触达 13/17 / 孤儿 02·12·15·16 | ipynb 实验四（映射依据 Domain Model §2 + Crosswalk） |
| W1-W6 判分 0/6 | PT-W4-D6《组装SemanticModel与验证设计》装配规则 × ontology 实测 |
| 熵增模拟双曲线 | ipynb 实验五（情景假设显式标注） |
